# 05: Collect cp_measure results

In [62]:
import spatialdata as sd
import spatialdata_plot
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from spatialdata.models import TableModel, Labels2DModel
import anndata as ad
import xarray as xr
import dask.array as da
import numpy as np

In [63]:
proj_dir = Path("/lustre/groups/ml01/projects/2025_ttreis_icml_cpmeasure")
data_dir = proj_dir / Path("data")

In [64]:
sdata_sample1 = sd.read_zarr(data_dir / "xenium_sample1_subset_baked_cropped.zarr")
sdata_sample2 = sd.read_zarr(data_dir / "xenium_sample2_subset_baked_cropped.zarr")

version mismatch: detected: RasterFormatV02, requested: FormatV04
/ictstr01/home/icb/tim.treis/miniconda/envs/cpmeasure_icml/lib/python3.11/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04
/ictstr01/home/icb/tim.treis/miniconda/envs/cpmeasure_icml/lib/python3.11/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04


In [65]:
sample1_morpho_core = pd.read_csv(data_dir / Path("xenium_sample1_subset_morpho_core.csv"), index_col=0)
sample1_morpho_cpm = pd.read_csv(data_dir / Path("xenium_sample1_subset_morpho_cpm.csv"), index_col=0)
sample1_morpho_ski = pd.read_csv(data_dir / Path("xenium_sample1_subset_morpho_ski.csv"), index_col=0)


sample1_morpho_df = pd.concat([
    sample1_morpho_core,
    sample1_morpho_cpm,
    sample1_morpho_ski,
], axis=1)

# adjust index to match sdata obj
sample1_morpho_df.index -= 1

sample1_morpho_df

,intensity_max_0,intensity_min_0,intensity_mean_0,intensity_std_0,RadialDistribution_FracAtD_1of4_0,RadialDistribution_MeanFrac_1of4_0,RadialDistribution_RadialCV_1of4_0,RadialDistribution_FracAtD_2of4_0,RadialDistribution_MeanFrac_2of4_0,RadialDistribution_RadialCV_2of4_0,...,intensity_max_0,intensity_min_0,intensity_std_1,intensity_max_1,intensity_mean_1,intensity_min_1,intensity_max_2,intensity_min_2,intensity_mean_2,intensity_std_2
0,251.0,150.0,235.746580,8.999885,0.385740,9.583002,0.534598,0.433272,2.813957,0.455308,...,251.0,150.0,32.232129,248.0,224.902986,70.0,255.0,145.0,235.155551,15.756031
1,255.0,152.0,230.211401,14.728286,0.367725,14.200193,0.545451,0.121693,1.294967,0.884245,...,255.0,152.0,51.614263,254.0,193.430456,58.0,255.0,129.0,221.412866,25.116285
28,251.0,147.0,222.722127,20.106363,0.345679,7.197531,1.387514,0.204586,1.217075,0.642602,...,251.0,147.0,24.118487,245.0,206.989708,121.0,250.0,185.0,234.773585,9.557635
32,252.0,129.0,220.912943,32.828724,0.135231,4.388256,0.675012,0.391459,3.387426,0.606819,...,252.0,129.0,42.635358,244.0,194.928351,88.0,249.0,167.0,228.348998,18.530966
59,255.0,95.0,222.837269,28.127057,0.404810,9.513026,0.513310,0.278557,1.775542,0.617801,...,255.0,95.0,22.231088,253.0,179.967033,81.0,255.0,160.0,228.898527,11.034814
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
574421,249.0,144.0,207.011699,21.829809,0.129103,3.254054,1.100880,0.008753,0.057551,0.494869,...,249.0,144.0,20.417286,215.0,165.436928,100.0,248.0,176.0,222.876907,12.493461
574457,255.0,106.0,227.787500,32.330265,0.048980,1.230185,0.600613,0.248980,1.848096,0.492704,...,255.0,106.0,30.065090,237.0,175.421759,78.0,249.0,153.0,224.805556,16.164237
574464,255.0,176.0,232.060224,21.052502,0.021016,0.535902,0.862098,0.353765,2.360640,0.385888,...,255.0,176.0,21.850201,242.0,173.528011,122.0,255.0,190.0,225.862745,10.993794
574489,253.0,112.0,228.534727,24.105809,0.284680,7.799688,0.707115,0.167672,1.255093,0.534003,...,253.0,112.0,35.381398,244.0,185.053863,75.0,253.0,164.0,227.420624,14.707612


In [66]:
# create matching idx

s1_morph_idx = sample1_morpho_df.index

sdata_sample1.tables["table_subset"].obs.index = sdata_sample1.tables["table_subset"].obs.index.astype(int)
s1_gexp_idx = sdata_sample1.tables["table_subset"].obs.index.astype(int)

s1_shared_idx = set(s1_morph_idx) & set(s1_gexp_idx)

In [67]:
# sample1

adata_sample1_morphology = ad.AnnData(
    sample1_morpho_df.loc[list(s1_shared_idx), :],
    obs=sdata_sample1.tables["table_subset"].obs.loc[list(s1_shared_idx), :]
)

adata_sample1_morphology

/ictstr01/home/icb/tim.treis/miniconda/envs/cpmeasure_icml/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/ictstr01/home/icb/tim.treis/miniconda/envs/cpmeasure_icml/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 60997 × 957
    obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'leiden', 'Adipocytes_score', 'B_cells_score', 'Breast_cancer_score', 'Breast_glandular_cells_score', 'Breast_myoepithelial_cells_score', 'Dendritic_cells_score', 'Endothelial_cells_score', 'Epithelial_cells_score', 'Fibroblasts_score', 'Immune_cells_score', 'Macrophages_score', 'Mast_cells_score', 'Monocytes_score', 'NK_cells_score', 'Smooth_muscle_cells_score', 'T_cells_score', 'predicted_celltype'

In [68]:
# sample2

sample2_morpho_core = pd.read_csv(data_dir / Path("xenium_sample2_subset_morpho_core.csv"), index_col=0)
sample2_morpho_cpm = pd.read_csv(data_dir / Path("xenium_sample2_subset_morpho_cpm.csv"), index_col=0)
sample2_morpho_ski = pd.read_csv(data_dir / Path("xenium_sample2_subset_morpho_ski.csv"), index_col=0)

sample2_morpho_df = pd.concat([
    sample2_morpho_core,
    sample2_morpho_cpm,
    sample2_morpho_ski,
], axis=1)

# adjust index to match sdata obj
sample2_morpho_df.index -= 1

sample2_morpho_df

,intensity_mean_0,intensity_min_0,intensity_std_0,intensity_max_0,RadialDistribution_FracAtD_1of4_0,RadialDistribution_MeanFrac_1of4_0,RadialDistribution_RadialCV_1of4_0,RadialDistribution_FracAtD_2of4_0,RadialDistribution_MeanFrac_2of4_0,RadialDistribution_RadialCV_2of4_0,...,intensity_std_0,intensity_max_0,intensity_std_1,intensity_max_1,intensity_mean_1,intensity_min_1,intensity_max_2,intensity_mean_2,intensity_std_2,intensity_min_2
0,242.788514,87.0,27.846205,255.0,0.410112,9.047283,0.432832,0.258427,1.589275,0.607900,...,27.846205,255.0,23.152877,236.0,166.314889,48.0,251.0,224.368769,15.301081,135.0
7,230.118679,92.0,37.711751,255.0,0.246753,10.868359,0.753633,0.085343,0.912106,0.946407,...,37.711751,255.0,38.310380,251.0,160.993464,37.0,254.0,218.078087,23.265531,130.0
10,246.918158,114.0,11.017954,255.0,0.146341,3.058704,0.609362,0.121951,0.718267,0.512784,...,11.017954,255.0,32.004898,247.0,169.315717,43.0,254.0,224.171088,13.815791,142.0
12,237.316708,128.0,12.042707,255.0,0.073903,1.741864,0.577421,0.182448,1.169601,0.701481,...,12.042707,255.0,27.579814,247.0,217.293904,61.0,253.0,235.269556,11.852563,149.0
23,234.884829,149.0,10.177089,255.0,0.100719,2.499636,0.753160,0.219424,1.454132,0.698718,...,10.177089,255.0,27.695133,247.0,215.742632,93.0,254.0,233.251278,11.714493,158.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
365576,209.816304,100.0,38.065189,255.0,0.313688,8.745247,0.695705,0.450570,3.558152,0.475842,...,38.065189,255.0,31.535895,248.0,159.961957,60.0,254.0,220.160870,17.888033,143.0
365580,200.097489,32.0,57.222383,255.0,0.268456,6.267068,0.805451,0.171141,1.093042,0.626305,...,57.222383,255.0,52.322407,247.0,149.141802,1.0,253.0,209.358936,34.234696,101.0
365588,222.854200,38.0,44.285260,255.0,0.238193,4.696869,0.785838,0.379877,2.261342,0.393000,...,44.285260,255.0,43.594298,253.0,167.569466,18.0,255.0,219.861595,24.972988,120.0
365601,220.820513,175.0,16.295674,247.0,0.057203,1.115466,0.635746,0.156780,0.853173,0.635110,...,16.295674,247.0,18.265771,218.0,180.384615,125.0,246.0,231.188034,9.048711,193.0


In [69]:
s2_gexp_obs = sdata_sample2.tables["table_subset"].obs.sort_values("cell_id").set_index("cell_id", inplace=False)
s2_gexp_obs.index.name = None
s2_gexp_obs["cell_id"] = s2_gexp_obs.index
s2_gexp_obs.index -= 1
s2_gexp_obs

,transcript_counts,control_probe_counts,control_codeword_counts,total_counts,cell_area,nucleus_area,region,sample,n_genes_by_counts,log1p_n_genes_by_counts,...,Fibroblasts_score,Immune_cells_score,Macrophages_score,Mast_cells_score,Monocytes_score,NK_cells_score,Smooth_muscle_cells_score,T_cells_score,predicted_celltype,cell_id
0,154,0,0,154.0,279.923594,23.074844,cell_circles,sample2,72,4.290459,...,0.084264,-0.175716,-0.118000,0.120529,-0.179646,-0.148942,-0.039396,-0.138726,Endothelial cells,1
7,122,0,0,122.0,131.269219,19.101094,cell_circles,sample2,49,3.912023,...,0.051658,-0.207091,0.002096,-0.015036,-0.161261,-0.090687,-0.057846,-0.119596,Endothelial cells,8
10,66,0,0,66.0,658.784531,10.882656,cell_circles,sample2,32,3.496508,...,0.217872,-0.224758,-0.078788,-0.033795,-0.181510,-0.061613,0.020480,-0.049669,Fibroblasts,11
12,120,0,0,120.0,497.034844,13.592031,cell_circles,sample2,38,3.663562,...,-0.038298,-0.137233,-0.060159,-0.162709,-0.114356,-0.141891,0.110626,-0.103762,Adipocytes,13
23,234,0,0,234.0,830.423438,45.923906,cell_circles,sample2,68,4.234107,...,-0.094749,-0.100935,-0.002139,-0.193077,-0.167866,-0.141482,0.089342,-0.110466,Adipocytes,24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
365576,94,0,0,94.0,83.087500,39.060156,cell_circles,sample2,50,3.931826,...,-0.130859,-0.354817,-0.049156,-0.036146,-0.064053,0.031939,-0.090804,0.045717,Breast glandular cells,365577
365580,16,0,0,16.0,30.570781,11.063281,cell_circles,sample2,15,2.772589,...,0.120108,-0.141683,0.260094,0.000000,0.661189,-0.024151,-0.097894,0.009752,Monocytes,365581
365588,50,0,0,50.0,85.480781,19.372031,cell_circles,sample2,32,3.496508,...,0.151603,-0.080796,0.220539,-0.026932,-0.026932,-0.041316,0.027673,0.133979,Macrophages,365589
365601,20,0,0,20.0,10.566562,6.276719,cell_circles,sample2,15,2.772589,...,-0.056762,-0.128768,-0.009658,0.000000,0.000000,-0.021949,-0.067183,-0.009376,Breast glandular cells,365602


In [70]:
# create matching idx

s2_morph_idx = sample2_morpho_df.index.astype(int)

s2_gexp_idx = s2_gexp_obs.index

s2_shared_idx = set(s2_morph_idx) & set(s2_gexp_idx)

In [71]:
# sample2

adata_sample2_morphology = ad.AnnData(
    sample2_morpho_df.loc[list(s2_shared_idx), :],
    obs=s2_gexp_obs.loc[list(s2_shared_idx), :]
)

adata_sample2_morphology

/ictstr01/home/icb/tim.treis/miniconda/envs/cpmeasure_icml/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/ictstr01/home/icb/tim.treis/miniconda/envs/cpmeasure_icml/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 52206 × 957
    obs: 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'leiden', 'Adipocytes_score', 'B_cells_score', 'Breast_cancer_score', 'Breast_glandular_cells_score', 'Breast_myoepithelial_cells_score', 'Dendritic_cells_score', 'Endothelial_cells_score', 'Epithelial_cells_score', 'Fibroblasts_score', 'Immune_cells_score', 'Macrophages_score', 'Mast_cells_score', 'Monocytes_score', 'NK_cells_score', 'Smooth_muscle_cells_score', 'T_cells_score', 'predicted_celltype', 'cell_id'

In [72]:
adata_sample1_morphology.write(data_dir / "xenium_sample1_subset_morphology.h5ad")

In [73]:
adata_sample2_morphology.write(data_dir / "xenium_sample2_subset_morphology.h5ad")